# Sinmate: QLoRA Fine-Tuning Gemma-3-1B-IT for Spoken Sinhala

This Google Colab notebook fine-tunes **`google/gemma-3-1b-it`** using **4-bit QLoRA** on a conversational Sinhala dataset (*Kathitha Bhashawa* with Sri Lankan fillers, Singlish, and 4 personas).

### Pipeline Overview:
1. **Load Base Model**: `google/gemma-3-1b-it` quantized to 4-bit (VRAM footprint ~1.8GB on free Colab T4).
2. **Chat Template**: Standard Gemma turn tags `<start_of_turn>user...<end_of_turn>`.
3. **Train with QLoRA**: LoRA adapters on linear attention and MLP layers (`r=16, alpha=32`).
4. **Inference Test**: Multi-persona conversational test in colloquial Sinhala.
5. **Export GGUF**: Export 4-bit quantized GGUF weights (~700MB–800MB) for offline laptop execution.

In [1]:
# 1. Install fine-tuning dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

In [2]:
# 2. Hugging Face Login (Access Gemma license)
# Token lookup order: Colab secret -> HF_TOKEN env var -> getpass prompt
import os
from getpass import getpass
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass  # not on Colab, or no HF_TOKEN secret / notebook access

token = token or os.environ.get('HF_TOKEN') or getpass('Enter your Hugging Face access token: ')
login(token=token)

In [4]:
# 3. Clone or upload Sinmate dataset
import os
if not os.path.exists("data"):
    !git clone https://github.com/chathuwa-whiz/Sinmate.git
    %cd Sinmate

from datasets import load_dataset
dataset = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
print("Dataset loaded:", dataset)

Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 106
    })
    val: Dataset({
        features: ['id', 'persona', 'input_type', 'emotion', 'scenario', 'messages', 'category'],
        num_rows: 11
    })
})


In [5]:
# 4. Load Gemma-3-1B-IT in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-1b-it"

# bf16 needs compute capability >= 8.0 (A100/L4/...). The free-tier T4 is 7.5: bf16 kernels fail there,
# and Gemma 3 overflows in fp16, so fall back to fp32 compute (fine for a 1B model).
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print(f"GPU: {torch.cuda.get_device_name()} | compute dtype: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

GPU: Tesla T4 | compute dtype: torch.float32


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [ ]:
# 5. Format Chat Templates & Train with Response-Only Loss Masking
from trl import SFTConfig, SFTTrainer

# Ensure tokenizer padding token is defined for batched training
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Use conversational messages format directly (TRL applies chat template automatically)
train_dataset = dataset["train"].select_columns(["messages"])
val_dataset = dataset["val"].select_columns(["messages"])

# TRL v0.14+ natively supports assistant_only_loss in SFTConfig for conversational datasets.
# In TRL >= 0.20, DataCollatorForCompletionOnlyLM has been deprecated and removed.
try:
    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=0.05,  # float < 1 = ratio of total steps
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=3,
        fp16=False,
        bf16=USE_BF16,  # set in cell 4; False on T4 (fp32 compute)
        save_strategy="epoch",
        eval_strategy="epoch",
        max_length=512,
        assistant_only_loss=True,  # Response-only loss: computes loss ONLY on assistant turns
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=training_args,
    )
except (TypeError, ValueError):
    # Fallback for legacy TRL versions (< 0.14) that required DataCollatorForCompletionOnlyLM
    from trl import DataCollatorForCompletionOnlyLM

    def format_prompts(example):
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

    train_dataset = dataset["train"].map(format_prompts, remove_columns=dataset["train"].column_names)
    val_dataset = dataset["val"].map(format_prompts, remove_columns=dataset["val"].column_names)

    collator = DataCollatorForCompletionOnlyLM(
        response_template="<start_of_turn>model\n",
        instruction_template="<start_of_turn>user\n",
        tokenizer=tokenizer,
    )

    training_args = SFTConfig(
        output_dir="./gemma3_sinhala_output",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=0.05,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        logging_steps=10,
        num_train_epochs=3,
        fp16=False,
        bf16=USE_BF16,
        save_strategy="epoch",
        eval_strategy="epoch",
        dataset_text_field="text",
        max_length=512,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collator,
        processing_class=tokenizer,
        args=training_args,
    )

trainer.train()


In [15]:
# 6. Test Interactive Multi-Persona Inference
test_messages = [
    {
        "role": "system",
        "content": "You are Sinmate, an affectionate, emotionally intelligent native Sinhala companion sitting right beside the user.\n[ROLE: BEST_FRIEND]\nYou are the user's closest best friend. Responses must strictly be 1-2 spoken colloquial sentences with Sri Lankan fillers."
    },
    {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, top_p=0.9, do_sample=True)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n🤖 SINMATE RESPONSE:", response)



🤖 SINMATE RESPONSE: හරි මචි, ගහ ගහ. ඒක හරිම හොඳ උනන්දුව.


In [16]:
# 7. Save LoRA Adapters
model.save_pretrained("sinmate-gemma-3-1b-lora")
tokenizer.save_pretrained("sinmate-gemma-3-1b-lora")
print("Adapters saved successfully to sinmate-gemma-3-1b-lora!")

Adapters saved successfully to sinmate-gemma-3-1b-lora!


In [ ]:
# 8. Merge LoRA with Base Model for GGUF Export
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

print("Loading base model in FP16 to merge LoRA...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
)
merged_model = PeftModel.from_pretrained(base_model, "sinmate-gemma-3-1b-lora")
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("sinmate-gemma-3-1b-merged")
tokenizer.save_pretrained("sinmate-gemma-3-1b-merged")
print("Merged model saved successfully to ./sinmate-gemma-3-1b-merged!")


In [ ]:
# 9. Convert Merged Model to 4-bit GGUF via llama.cpp
import os
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggml-org/llama.cpp.git
!pip install -r llama.cpp/requirements.txt

# Convert HuggingFace checkpoint to FP16 GGUF
!python llama.cpp/convert_hf_to_gguf.py sinmate-gemma-3-1b-merged --outfile sinmate-gemma-3-1b-f16.gguf --outtype f16

# Build llama-quantize and quantize to Q4_K_M (4-bit)
!cmake -B llama.cpp/build llama.cpp && cmake --build llama.cpp/build --config Release -j --target llama-quantize
!./llama.cpp/build/bin/llama-quantize sinmate-gemma-3-1b-f16.gguf sinmate-gemma-3-1b-q4_k_m.gguf Q4_K_M
print("GGUF quantization complete: sinmate-gemma-3-1b-q4_k_m.gguf")


In [ ]:
# 10. Generate Modelfile for Ollama
modelfile = '''FROM ./sinmate-gemma-3-1b-q4_k_m.gguf
TEMPLATE """{{ if .System }}<start_of_turn>system
{{ .System }}<end_of_turn>
{{ end }}{{ if .Prompt }}<start_of_turn>user
{{ .Prompt }}<end_of_turn>
{{ end }}<start_of_turn>model
{{ .Response }}<end_of_turn>
"""
PARAMETER stop "<start_of_turn>"
PARAMETER stop "<end_of_turn>"
PARAMETER temperature 0.7
PARAMETER top_p 0.9
'''

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)

print("Modelfile written! To import into Ollama locally:")
print("1. Download sinmate-gemma-3-1b-q4_k_m.gguf and Modelfile")
print("2. Run: ollama create sinmate-gemma3:1b -f Modelfile")
